In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

synthetic_df = pd.read_csv(
    "data/Synthetic_data/synthetic_trade_history.csv"
)

real_df = pd.read_csv(
    "data/CME_Swap_data/cme_gbp_swaps_clean.csv"
)

synthetic_df["trade_timestamp"] = pd.to_datetime(
    synthetic_df["trade_timestamp"],
    dayfirst=True
)

print("Synthetic records:", len(synthetic_df))
print("Real CME records:", len(real_df))

Synthetic records: 2000
Real CME records: 28


In [2]:
## Define common features
FEATURES = ["notional_amount", "tenor_years"]
TARGET = "simulated_rejected"

# Check required inputs
assert all(col in synthetic_df.columns for col in FEATURES + [TARGET])
assert all(col in real_df.columns for col in FEATURES)

# Check numeric values
for df in [synthetic_df, real_df]:
    for col in FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")

assert synthetic_df[FEATURES].notna().all().all()
assert real_df[FEATURES].notna().all().all()

print("Common features verified:", FEATURES)

Common features verified: ['notional_amount', 'tenor_years']


In [3]:
## Recreate the chronological split
trade_order = (
    synthetic_df.groupby("trade_id")["trade_timestamp"]
    .min()
    .sort_values(kind="stable")
)

n = len(trade_order)

train_ids = set(trade_order.index[:int(n * 0.70)])
val_ids = set(trade_order.index[int(n * 0.70):int(n * 0.85)])
test_ids = set(trade_order.index[int(n * 0.85):])

train_df = synthetic_df[
    synthetic_df["trade_id"].isin(train_ids)
].copy()

val_df = synthetic_df[
    synthetic_df["trade_id"].isin(val_ids)
].copy()

test_df = synthetic_df[
    synthetic_df["trade_id"].isin(test_ids)
].copy()

# Split integrity
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)

assert train_df["trade_timestamp"].max() <= val_df["trade_timestamp"].min()
assert val_df["trade_timestamp"].max() <= test_df["trade_timestamp"].min()

X_train = train_df[FEATURES]
y_train = train_df[TARGET]

X_val = val_df[FEATURES]
y_val = val_df[TARGET]

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Reserved test records:", len(test_df))

Training: 1400
Validation: 300
Reserved test records: 300


In [4]:
## Train two-feature Logistic Regression
real_compatible_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

real_compatible_model.fit(X_train, y_train)

val_probs = real_compatible_model.predict_proba(X_val)[:, 1]

print(
    "Validation PR-AUC:",
    round(average_precision_score(y_val, val_probs), 4)
)

Validation PR-AUC: 0.202


In [5]:
## Select threshold using validation data
threshold_results = []

for t in np.arange(0.01, 1.0, 0.01):
    pred = (val_probs >= t).astype(int)

    threshold_results.append({
        "Threshold": round(t, 2),
        "Precision": precision_score(
            y_val, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_val, pred, zero_division=0
        ),
        "F1": f1_score(
            y_val, pred, zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

best = threshold_df.loc[threshold_df["F1"].idxmax()]
real_threshold = float(best["Threshold"])

print("Selected threshold:", real_threshold)
print(best.round(4))


Selected threshold: 0.12
Threshold    0.1200
Precision    0.2593
Recall       0.4118
F1           0.3182
Name: 11, dtype: float64


In [6]:
## Check real CME data distribution
for col in FEATURES:
    train_min = X_train[col].min()
    train_max = X_train[col].max()

    real_df[f"{col}_out_of_range"] = (
        (real_df[col] < train_min) |
        (real_df[col] > train_max)
    )

    print(f"\n{col}")
    print("Training range:", train_min, "to", train_max)
    print("CME range:", real_df[col].min(), "to", real_df[col].max())
    print(
        "CME records outside training range:",
        real_df[f"{col}_out_of_range"].sum()
    )


notional_amount
Training range: 1000000 to 100000000
CME range: 600000 to 1800000000
CME records outside training range: 8

tenor_years
Training range: 1 to 20
CME range: 0.11 to 10.0
CME records outside training range: 4


In [7]:
## Predict on real CME transactions
real_probs = real_compatible_model.predict_proba(
    real_df[FEATURES]
)[:, 1]

real_df["predicted_risk"] = real_probs

real_df["risk_flag"] = np.where(
    real_df["predicted_risk"] >= real_threshold,
    "Review Required",
    "Below Threshold"
)

display(real_df[
    [
        "trade_id",
        "notional_amount",
        "tenor_years",
        "predicted_risk",
        "risk_flag",
        "notional_amount_out_of_range",
        "tenor_years_out_of_range"
    ]
].head(10))

print("\nRisk classification:")
print(real_df["risk_flag"].value_counts())

,trade_id,notional_amount,tenor_years,predicted_risk,risk_flag,notional_amount_out_of_range,tenor_years_out_of_range
0,SDRIR91238342,50000000,2.0,0.072323,Below Threshold,False,False
1,SDRIR91238333,12000000,10.0,0.072340,Below Threshold,False,False
2,SDRIR91237936,600000,10.0,0.064622,Below Threshold,True,False
3,SDRIR91237927,50000000,2.0,0.072323,Below Threshold,False,False
4,SDRIR91237922,6000000,2.0,0.046576,Below Threshold,False,False
5,SDRIR91237940,17000000,2.0,0.052049,Below Threshold,False,False
6,SDRIR91237934,12000000,10.0,0.072340,Below Threshold,False,False
7,SDRIR91237928,3000000,2.0,0.045181,Below Threshold,False,False
8,SDRIR91237923,1000000,10.0,0.064879,Below Threshold,False,False
9,SDRIR91237941,5000000,10.0,0.067505,Below Threshold,False,False



Risk classification:
risk_flag
Below Threshold    24
Review Required     4
Name: count, dtype: int64


In [8]:
real_df["out_of_training_range"] = (
    real_df["notional_amount_out_of_range"]
    | real_df["tenor_years_out_of_range"]
)

print("Unique records outside training range:",
      real_df["out_of_training_range"].sum())

print("\nRisk flags vs training-range compatibility:")
print(pd.crosstab(
    real_df["risk_flag"],
    real_df["out_of_training_range"]
))

Unique records outside training range: 8

Risk flags vs training-range compatibility:
out_of_training_range  False  True 
risk_flag                          
Below Threshold           20      4
Review Required            0      4


## Real CME Compatibility — Findings

- Real CME GBP OIS transactions tested: 28
- Successfully scored: 28
- Below threshold: 24
- Review required: 4
- Outside synthetic training ranges: 8
- All 4 review alerts were outside training ranges.
- No within-range trades were flagged.

### Conclusion
The two-feature Logistic Regression pipeline successfully
processed real CME transactions.

However, flagged trades were associated with out-of-range
feature values, highlighting a potential distribution-shift issue.

Actual regulatory rejection labels were unavailable.
Therefore, prediction accuracy, precision and recall
cannot be verified using this dataset.

This experiment demonstrates real-data inference compatibility,
not validated real-world rejection-risk prediction.